# B2 v2 — WavLM-CTC Fine-tuning on TORGO (B1-initialized, UASpeech validation)

**Changes from original B2:**
1. Loads B1 weights (CTC head pretrained on LibriSpeech) instead of random init
2. Uses UASpeech samples as validation set (prevents overfitting to TORGO prompts)
3. Previous TORGO validation utterances are moved back into training

**Pipeline:** B1 (head pretrained on LibriSpeech) → B2 (full fine-tune on TORGO)

In [2]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

os.environ["HF_HOME"]           = "/kaggle/temp/hf"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/temp/hf/datasets"
os.environ["HF_HUB_CACHE"]      = "/kaggle/temp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/hf/transformers"
os.environ["XDG_CACHE_HOME"]    = "/kaggle/temp/.cache"

for p in [os.environ["HF_HOME"], os.environ["HF_DATASETS_CACHE"],
          os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"],
          os.environ["XDG_CACHE_HOME"]]:
    os.makedirs(p, exist_ok=True)

!rm -rf /kaggle/working/*
print("Cache dirs ready.")

Cache dirs ready.


In [3]:
!pip -q install -U transformers datasets accelerate evaluate jiwer soundfile packaging

In [4]:
import numpy as np
import pandas as pd
import torch
import evaluate
import random
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from itertools import groupby
from collections import Counter

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import (
    WavLMForCTC,
    Wav2Vec2Processor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    EarlyStoppingCallback,
)
import transformers
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

transformers: 5.5.0
torch: 2.10.0+cu128
GPU: True
GPU name: Tesla T4


In [5]:
!python /kaggle/input/datasets/kavishk/train-test-splits/create_splits.py

╔══════════════════════════════════════════════════════╗
║         Dataset Split Creator                       ║
║         TORGO + UASpeech                            ║
╚══════════════════════════════════════════════════════╝

TORGO Split
Total utterances: 16552

Train: 5578 utterances
Test:  1798 utterances

Train severity distribution:
  control: 1500
  mild: 810
  moderate: 1087
  severe: 2181

Test severity distribution:
  control: 302
  mild: 673
  moderate: 587
  severe: 236

Train speakers: ['F03', 'FC02', 'FC03', 'M01', 'M02', 'M03', 'M04', 'MC01', 'MC02', 'MC03', 'MC04']
Test speakers:  ['F01', 'F04', 'FC01', 'M05']
Saving the dataset (2/2 shards): 100%|█| 5578/5578 [00:02<00:00, 2679.39 example
Saving the dataset (1/1 shards): 100%|█| 1798/1798 [00:00<00:00, 2237.08 example

Saved: /kaggle/working/torgo_train
Saved: /kaggle/working/torgo_test

UASpeech Splits
UASpeech train (full): 38656 utterances
UASpeech test (full):  18740 utterances

Stratified validation sampling (800 t

## Step 1 — Load pre-made splits

All splits created by `create_splits.py` and saved as a Kaggle Dataset.
Just load them — no split logic needed in experiment notebooks.

In [6]:
# ── Load all splits ──
from datasets import load_from_disk

# Adjust this base path to your Kaggle Dataset name
SPLIT_BASE = "/kaggle/working/"

torgo_train = load_from_disk(f"{SPLIT_BASE}/torgo_train")
torgo_test  = load_from_disk(f"{SPLIT_BASE}/torgo_test")
ua_val      = load_from_disk(f"{SPLIT_BASE}/ua_val")
ua_test     = load_from_disk(f"{SPLIT_BASE}/ua_test")

print(f"TORGO train:  {len(torgo_train)} utterances")
print(f"TORGO test:   {len(torgo_test)} utterances")
print(f"UASpeech val: {len(ua_val)} utterances")
print(f"UASpeech test:{len(ua_test)} utterances")
print(f"\nTrain speakers: {sorted(set(torgo_train['speaker']))}")
print(f"Test speakers:  {sorted(set(torgo_test['speaker']))}")

TORGO train:  5578 utterances
TORGO test:   1798 utterances
UASpeech val: 800 utterances
UASpeech test:18740 utterances

Train speakers: ['F03', 'FC02', 'FC03', 'M01', 'M02', 'M03', 'M04', 'MC01', 'MC02', 'MC03', 'MC04']
Test speakers:  ['F01', 'F04', 'FC01', 'M05']


## Step 3 — Load B1 model and processor

In [7]:
# ── Load B1 model (CTC head pretrained on LibriSpeech) ──
# Adjust path to wherever you saved B1
B1_MODEL_PATH = "/kaggle/input/datasets/kavishk/b1-wavlm-ctc-librispeech/kaggle/working/b1_wavlm_ctc_librispeech"

model = WavLMForCTC.from_pretrained(
    B1_MODEL_PATH,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
)
processor = Wav2Vec2Processor.from_pretrained(B1_MODEL_PATH)

# Freeze CNN feature encoder (standard practice)
model.freeze_feature_encoder()

# UNFREEZE the transformer encoder (this is the key difference from B1)
for param in model.wavlm.encoder.parameters():
    param.requires_grad = True

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Model loaded from: {B1_MODEL_PATH}")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {frozen_params:,}  (CNN feature encoder only)")
print(f"Vocab size:           {model.config.vocab_size}")
print(f"Pad token id:         {processor.tokenizer.pad_token_id}")

Loading weights:   0%|          | 0/250 [00:00<?, ?it/s]

Model loaded from: /kaggle/input/datasets/kavishk/b1-wavlm-ctc-librispeech/kaggle/working/b1_wavlm_ctc_librispeech
Total parameters:     94,406,544
Trainable parameters: 90,206,096
Frozen parameters:    4,200,448  (CNN feature encoder only)
Vocab size:           32
Pad token id:         29


## Step 4 — Preprocessing

In [8]:
# ── Filter long audio ──
MAX_AUDIO_SAMPLES = 240_000  # 15 seconds

def filter_long_torgo(example):
    return len(example["audio"]["array"]) <= MAX_AUDIO_SAMPLES

def filter_long_ua(example):
    return len(example["audio"]["array"]) <= MAX_AUDIO_SAMPLES

before = len(torgo_train)
torgo_train = torgo_train.filter(filter_long_torgo, num_proc=1)
print(f"TORGO train: {before} -> {len(torgo_train)} (filtered {before - len(torgo_train)})")

before = len(ua_val)
ua_val = ua_val.filter(filter_long_ua, num_proc=1)
print(f"UASpeech val: {before} -> {len(ua_val)} (filtered {before - len(ua_val)})")

Filter (num_proc=1):   0%|          | 0/5578 [00:00<?, ? examples/s]

TORGO train: 5578 -> 5545 (filtered 33)


Filter (num_proc=1):   0%|          | 0/800 [00:00<?, ? examples/s]

UASpeech val: 800 -> 797 (filtered 3)


In [9]:
# ── Preprocessing functions ──
def prepare_torgo(batch):
    audio = batch["audio"]
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_values"] = inputs.input_values[0]
    text = batch["transcription"].lower().strip()
    batch["labels"] = processor.tokenizer(text).input_ids
    return batch

def prepare_uaspeech(batch):
    audio = batch["audio"]
    inputs = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
    )
    batch["input_values"] = inputs.input_values[0]
    text = batch["text"].lower().strip()
    batch["labels"] = processor.tokenizer(text).input_ids
    return batch

print("Preprocessing functions defined.")

Preprocessing functions defined.


In [10]:
# ── Preprocess TORGO training data ──
torgo_remove = [c for c in torgo_train.column_names if c not in []]

train_p = torgo_train.map(
    prepare_torgo,
    remove_columns=torgo_train.column_names,
    num_proc=1,
    desc="Preprocessing TORGO train",
)

print(f"TORGO train preprocessed: {len(train_p)}")

# Sanity check
sample = train_p[0]
print(f"Sample input length: {len(sample['input_values'])}")
print(f"Sample labels: {sample['labels'][:20]}")
print(f"Sample decoded: {processor.tokenizer.decode(sample['labels'])}")

Preprocessing TORGO train (num_proc=1):   0%|          | 0/5545 [00:00<?, ? examples/s]

TORGO train preprocessed: 5545
Sample input length: 35680
Sample labels: [18, 11, 4, 4, 15]
Sample decoded: slep


In [11]:
# ── Preprocess UASpeech validation data ──
val_p = ua_val.map(
    prepare_uaspeech,
    remove_columns=ua_val.column_names,
    num_proc=1,
    desc="Preprocessing UASpeech val",
)

print(f"UASpeech val preprocessed: {len(val_p)}")

# Sanity check
sample = val_p[0]
print(f"Sample input length: {len(sample['input_values'])}")
print(f"Sample labels: {sample['labels']}")
print(f"Sample decoded: {processor.tokenizer.decode(sample['labels'])}")

Preprocessing UASpeech val (num_proc=1):   0%|          | 0/797 [00:00<?, ? examples/s]

UASpeech val preprocessed: 797
Sample input length: 53017
Sample labels: [0, 13, 24, 1, 14, 3, 24]
Sample decoded: anybody


## Step 5 — Data Collator

In [12]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = "longest"

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor)
print("Data collator ready.")

Data collator ready.


## Step 6 — Metrics and Callbacks

In [13]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def decode_ctc_predictions(pred_ids, tokenizer):
    blank_id = tokenizer.pad_token_id
    decoded = []
    for seq in pred_ids:
        collapsed = [k for k, _ in groupby(seq)]
        filtered = [t for t in collapsed if t != blank_id]
        if len(filtered) == 0:
            decoded.append("")
        else:
            decoded.append(tokenizer.decode(filtered, skip_special_tokens=True))
    return decoded

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    label_ids = pred.label_ids

    label_ids = np.where(
        label_ids != -100,
        label_ids,
        processor.tokenizer.pad_token_id
    )

    pred_str = decode_ctc_predictions(pred_ids, processor.tokenizer)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    pred_str = [p.strip() for p in pred_str]
    label_str = [l.strip() for l in label_str]

    eval_preds = [p if len(p) > 0 else "\u27e8empty\u27e9" for p in pred_str]
    eval_labels = [l if len(l) > 0 else "\u27e8empty\u27e9" for l in label_str]

    wer = wer_metric.compute(predictions=eval_preds, references=eval_labels)
    cer = cer_metric.compute(predictions=eval_preds, references=eval_labels)

    return {"wer": round(wer, 4), "cer": round(cer, 4)}

print("Metrics ready.")

Metrics ready.


In [14]:
class PrintLossCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if "loss" in logs:
            print(f"[step {state.global_step}] train_loss={logs['loss']:.4f}")
        if "learning_rate" in logs:
            print(f"[step {state.global_step}] lr={logs['learning_rate']:.2e}")
        if "eval_cer" in logs:
            print(f"[step {state.global_step}] val_cer={logs['eval_cer']:.4f}  val_wer={logs.get('eval_wer', 'N/A')}")

class SanityCheckCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        model.eval()
        device = next(model.parameters()).device
        try:
            sample = val_p[0]
            input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = model(input_values=input_values).logits
            pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
            pred_text = decode_ctc_predictions(pred_ids, processor.tokenizer)[0]
            label_ids = [x for x in sample["labels"] if x != -100]
            ref_text = processor.tokenizer.decode(label_ids, skip_special_tokens=True)
            print(f"  [SANITY CHECK - UASpeech val] REF : {ref_text}")
            print(f"  [SANITY CHECK - UASpeech val] PRED: {pred_text}")
        except Exception as e:
            print(f"  [SANITY CHECK] Error: {e}")

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.001,
)

print("Callbacks ready.")

Callbacks ready.


## Step 7 — Training

In [15]:
CHECKPOINT_DIR = "/kaggle/working/b2v2_wavlm_ctc"

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,

    num_train_epochs=40,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch = 16
    learning_rate=5e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.005,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,

    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,

    logging_steps=25,
    logging_dir="/kaggle/temp/tb_logs",
    report_to="none",

    save_total_limit=3,
)

print("Training arguments set.")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set.
Effective batch size: 16


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_p,
    eval_dataset=val_p,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[PrintLossCallback(), SanityCheckCallback(), early_stopping],
)

print("\n--- DISK USAGE BEFORE TRAINING ---")
!du -sh /kaggle/working/* 2>/dev/null || true

print("\nStarting B2v2 training (B1-initialized, UASpeech validation)...")
trainer.train()


--- DISK USAGE BEFORE TRAINING ---
4.0K	/kaggle/working/b2v2_wavlm_ctc
4.0K	/kaggle/working/split_metadata.json
201M	/kaggle/working/torgo_test
1.6G	/kaggle/working/torgo_train
1.8G	/kaggle/working/ua_test
241M	/kaggle/working/ua_val

Starting B2v2 training (B1-initialized, UASpeech validation)...


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Epoch,Training Loss,Validation Loss,Wer,Cer
1,8.495100,2.941308,1.020100,0.766900
2,6.365187,2.902907,1.016300,0.775700
3,5.538690,3.221375,1.033900,0.740000
4,4.395520,3.176456,1.031400,0.734100
5,3.492777,3.522376,1.036400,0.740300
6,3.046284,3.466875,1.026300,0.733700
7,2.671859,4.105315,1.051400,0.742200
8,2.341183,3.511326,1.040200,0.718800
9,1.806311,4.123143,1.079000,0.745000
10,1.690866,4.250894,1.031400,0.760800


[step 25] train_loss=29.0948
[step 25] lr=8.65e-07
[step 50] train_loss=20.2338
[step 50] lr=1.77e-06
[step 75] train_loss=13.0712
[step 75] lr=2.67e-06
[step 100] train_loss=12.2731
[step 100] lr=3.57e-06
[step 125] train_loss=11.4158
[step 125] lr=4.47e-06
[step 150] train_loss=11.0038
[step 150] lr=5.37e-06
[step 175] train_loss=10.0297
[step 175] lr=6.27e-06
[step 200] train_loss=9.4271
[step 200] lr=7.17e-06
[step 225] train_loss=9.2125
[step 225] lr=8.07e-06
[step 250] train_loss=8.7038
[step 250] lr=8.97e-06
[step 275] train_loss=8.8043
[step 275] lr=9.87e-06
[step 300] train_loss=8.4209
[step 300] lr=1.08e-05
[step 325] train_loss=8.4951
[step 325] lr=1.17e-05
[step 347] val_cer=0.7669  val_wer=1.0201
  [SANITY CHECK - UASpeech val] REF : anybody
  [SANITY CHECK - UASpeech val] PRED: anb


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


[step 5225] train_loss=1.2749
[step 5225] lr=3.46e-05
[step 5250] train_loss=0.9735
[step 5250] lr=3.45e-05
[step 5275] train_loss=1.0856
[step 5275] lr=3.44e-05
[step 5300] train_loss=1.0408
[step 5300] lr=3.43e-05
[step 5325] train_loss=1.2069
[step 5325] lr=3.42e-05
[step 5350] train_loss=1.1953
[step 5350] lr=3.41e-05
[step 5375] train_loss=1.1412
[step 5375] lr=3.40e-05
[step 5400] train_loss=0.9786
[step 5400] lr=3.39e-05
[step 5425] train_loss=1.5477
[step 5425] lr=3.38e-05
[step 5450] train_loss=1.1094
[step 5450] lr=3.37e-05
[step 5475] train_loss=0.9947
[step 5475] lr=3.36e-05
[step 5500] train_loss=1.1239
[step 5500] lr=3.35e-05
[step 5525] train_loss=0.9227
[step 5525] lr=3.34e-05
[step 5550] train_loss=1.1053
[step 5550] lr=3.33e-05
[step 5552] val_cer=0.8054  val_wer=1.1606
  [SANITY CHECK - UASpeech val] REF : anybody
  [SANITY CHECK - UASpeech val] PRED: in


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5552, training_loss=3.3346708854955622, metrics={'train_runtime': 4495.6289, 'train_samples_per_second': 49.337, 'train_steps_per_second': 3.087, 'total_flos': 4.2439350660241807e+18, 'train_loss': 3.3346708854955622, 'epoch': 16.0})

In [17]:
# ── Save ──
FINAL_MODEL_PATH = "/kaggle/working/b2v2_wavlm_ctc_final"
trainer.save_model(FINAL_MODEL_PATH)
processor.save_pretrained(FINAL_MODEL_PATH)

print(f"Best model saved to {FINAL_MODEL_PATH}")

print("\n--- DISK USAGE ---")
!du -sh /kaggle/working/* 2>/dev/null || true

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best model saved to /kaggle/working/b2v2_wavlm_ctc_final

--- DISK USAGE ---
3.1G	/kaggle/working/b2v2_wavlm_ctc
361M	/kaggle/working/b2v2_wavlm_ctc_final
4.0K	/kaggle/working/split_metadata.json
201M	/kaggle/working/torgo_test
1.6G	/kaggle/working/torgo_train
1.8G	/kaggle/working/ua_test
241M	/kaggle/working/ua_val


## Step 8 — Evaluate on TORGO test set

In [18]:
# ── Preprocess TORGO test set ──
torgo_test_filtered = torgo_test.filter(filter_long_torgo, num_proc=1)

# Need to keep Category for per-severity eval
test_categories = torgo_test_filtered["Category"]

test_p = torgo_test_filtered.map(
    prepare_torgo,
    remove_columns=torgo_test_filtered.column_names,
    num_proc=1,
    desc="Preprocessing TORGO test",
)

print(f"TORGO test preprocessed: {len(test_p)}")

Filter (num_proc=1):   0%|          | 0/1798 [00:00<?, ? examples/s]

Preprocessing TORGO test (num_proc=1):   0%|          | 0/1786 [00:00<?, ? examples/s]

TORGO test preprocessed: 1786


In [19]:
# ── Evaluate on TORGO test ──
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds = []
all_labels = []
all_cats = []

TEST_BATCH_SIZE = 8

for i in range(0, len(test_p), TEST_BATCH_SIZE):
    end = min(i + TEST_BATCH_SIZE, len(test_p))

    batch = data_collator([
        {
            "input_values": test_p[j]["input_values"],
            "labels":       test_p[j]["labels"],
        }
        for j in range(i, end)
    ])

    input_values = batch["input_values"].to(device)
    attention_mask = batch["attention_mask"].to(device) if "attention_mask" in batch else None

    with torch.no_grad():
        logits = model(
            input_values,
            attention_mask=attention_mask
        ).logits

    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    pred_str = decode_ctc_predictions(pred_ids, processor.tokenizer)

    label_ids = batch["labels"].numpy()
    label_ids = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    all_preds.extend([p.strip() for p in pred_str])
    all_labels.extend([l.strip() for l in label_str])
    all_cats.extend([test_categories[j] for j in range(i, end)])

print(f"Decoded {len(all_preds)} TORGO test utterances.")

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


Decoded 1786 TORGO test utterances.


In [20]:
# ── TORGO overall metrics ──
eval_preds = [p if len(p) > 0 else "\u27e8empty\u27e9" for p in all_preds]

overall_wer = wer_metric.compute(predictions=eval_preds, references=all_labels)
overall_cer = cer_metric.compute(predictions=eval_preds, references=all_labels)

print("=" * 55)
print("B2v2 — WavLM-CTC (B1 init) — TORGO TEST RESULTS")
print("=" * 55)
print(f"Overall WER: {overall_wer:.4f}  ({overall_wer*100:.2f}%)")
print(f"Overall CER: {overall_cer:.4f}  ({overall_cer*100:.2f}%)")

# Per-severity
results_df = pd.DataFrame({
    "prediction": all_preds,
    "reference": all_labels,
    "Category": all_cats,
})

print("\nPer-severity results:")
print("-" * 55)

for cat in ["control", "mild", "moderate", "severe"]:
    subset = results_df[results_df["Category"] == cat]
    subset = subset[subset["reference"].str.strip().str.len() > 0]
    if len(subset) == 0:
        print(f"{cat:15s}: no samples")
        continue
    preds = [p if len(p) > 0 else "\u27e8empty\u27e9" for p in subset["prediction"].tolist()]
    cat_wer = wer_metric.compute(predictions=preds, references=subset["reference"].tolist())
    cat_cer = cer_metric.compute(predictions=preds, references=subset["reference"].tolist())
    print(f"{cat:15s}: WER={cat_wer*100:.2f}%  CER={cat_cer*100:.2f}%  (n={len(subset)})")

B2v2 — WavLM-CTC (B1 init) — TORGO TEST RESULTS
Overall WER: 0.4422  (44.22%)
Overall CER: 0.2215  (22.15%)

Per-severity results:
-------------------------------------------------------
control        : WER=17.79%  CER=5.92%  (n=302)
mild           : WER=22.27%  CER=7.11%  (n=673)
moderate       : WER=75.88%  CER=42.47%  (n=575)
severe         : WER=70.57%  CER=42.74%  (n=236)


In [21]:
# ── Sample predictions ──
print("\nSample TORGO test predictions:")
print("-" * 70)
for i in range(min(15, len(all_preds))):
    print(f"[{all_cats[i]:10s}] REF : {all_labels[i]}")
    print(f"           PRED: {all_preds[i]}")
    print()


Sample TORGO test predictions:
----------------------------------------------------------------------
[control   ] REF : alpha
           PRED: alpha

[control   ] REF : the
           PRED: the

[control   ] REF : except in the winter when the oze or snow or ice prevents
           PRED: exept in the winter when the ose or snow or ice prevent

[control   ] REF : raid
           PRED: raid

[control   ] REF : read
           PRED: lead

[control   ] REF : stuble
           PRED: stuble

[control   ] REF : ate
           PRED: eat

[control   ] REF : store
           PRED: store

[control   ] REF : sip
           PRED: sip

[control   ] REF : wish
           PRED: wish

[control   ] REF : slay
           PRED: slay

[control   ] REF : sigh
           PRED: sigh

[control   ] REF : jaged
           PRED: jacked

[control   ] REF : up
           PRED: up

[control   ] REF : chair
           PRED: chair



In [23]:
# ── Save results ──
results_df.to_csv("/kaggle/working/b2v2_torgo_test_results.csv", index=False)

summary = {
    "model": "B2v2_WavLM_CTC_B1init",
    "b1_pretrained_on": "LibriSpeech-clean-100",
    "b2_finetuned_on": "TORGO",
    "validation_set": "UASpeech (stratified sample)",
    "torgo_overall_wer": round(overall_wer, 4),
    "torgo_overall_cer": round(overall_cer, 4),
    "test_speakers": ["F01", "F04", "FC01", "M05"],
}
with open("/kaggle/working/b2v2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Results and summary saved.")

print("\n" + "=" * 55)
print("B2 results")
print("=" * 55)
print(f"{'':15s} {'WER':>10s} {'CER':>10s}")
print(f"{'B2 (B1 init)':15s} {overall_wer*100:9.2f}% {overall_cer*100:9.2f}%")

Results and summary saved.

B2 results
                       WER        CER
B2 (B1 init)        44.22%     22.15%


In [24]:
# ── Zip for download ──
!zip -r /kaggle/working/b2v2_wavlm_ctc_final.zip /kaggle/working/b2v2_wavlm_ctc_final/
print("Zip ready for download.")

  adding: kaggle/working/b2v2_wavlm_ctc_final/ (stored 0%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/added_tokens.json (deflated 20%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/training_args.bin (deflated 53%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/config.json (deflated 66%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/vocab.json (deflated 56%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/processor_config.json (deflated 44%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/model.safetensors (deflated 9%)
  adding: kaggle/working/b2v2_wavlm_ctc_final/tokenizer_config.json (deflated 73%)
Zip ready for download.
